# Fase 3 — Feature Engineering
## Progetto AI_microtradingAUX

Questo notebook documenta in dettaglio l'implementazione della Fase 3 del progetto, dedicata alla costruzione causale delle feature di mercato per l'agente di trading algoritmico su XAU/USD M1.

**Durata indicativa:** 3-4 giorni

**Obiettivo:** trasformare il dataset OHLCV validato nella Fase 1 in un set di feature numeriche pronte per il training dell'agente RL, garantendo che nessuna informazione futura venga usata per calcolare feature relative al passato (assenza di leakage temporale).

### Output prodotti

| File | Ruolo |
|---|---|
| `src/gold_rl/data/features.py` | Calcolo causale delle feature, warm-up, normalizzazione, filtro correlazione |
| `src/gold_rl/data/leakage_checks.py` | Test di invarianza future-append |
| `tests/test_features.py` | Test unitari sulle feature |
| `tests/test_no_leakage.py` | Test di non-leakage temporale |
| `scripts/build_features.py` | Pipeline end-to-end: caricamento → split → feature → normalizzazione → filtro correlazione → salvataggio |

### Criterio di completamento

> Aggiungere dati futuri non deve modificare le feature già costruite per il passato.

Questo criterio è stato verificato sia su dati sintetici (test automatici) sia concettualmente sui dati reali tramite l'architettura causale del codice (nessun `shift` negativo, nessuna finestra centrata, nessuna statistica calcolata sull'intero dataset).


## 1. Selezione delle feature

La prima selezione richiesta comprende 15 feature, raggruppabili in quattro categorie:

**Ritorni a diversi orizzonti**
- `return_1`, `return_4`, `return_16`: variazione percentuale del prezzo di chiusura rispetto a 1, 4 e 16 barre precedenti.

**Volatilità e posizione rispetto alle medie mobili**
- `atr_over_close`: Average True Range normalizzato sul prezzo di chiusura.
- `close_ema20_over_atr`, `close_ema50_over_atr`: distanza tra prezzo e media esponenziale, scalata per la volatilità (ATR).
- `ema20_ema50_over_atr`: distanza tra le due medie esponenziali, scalata per l'ATR.

**Struttura della candela**
- `candle_body_over_range`: corpo della candela (close-open) diviso per l'ampiezza high-low.
- `upper_wick_over_range`, `lower_wick_over_range`: ombre superiore e inferiore normalizzate sull'ampiezza della candela.

**Volume e tempo**
- `relative_volume`: volume della barra corrente rispetto alla media mobile del volume.
- `hour_sin`, `hour_cos`: codifica ciclica dell'ora del giorno (evita la discontinuità 23:59 → 00:00).

**Stato dell'agente (feature di contesto, non di mercato)**
- `current_position`, `unrealized_return`: posizione corrente e P&L non realizzato dell'agente, che dovranno essere popolati dall'Execution Engine nelle fasi successive (per ora sono stub a zero).


## 2. Calcolo causale delle feature

### Principio guida

Ogni feature deve dipendere **esclusivamente** dalla barra corrente e da barre passate. Questo esclude:
- `shift(-n)` (guarda avanti nel tempo),
- medie o finestre centrate (`center=True` in `rolling`),
- qualunque statistica di normalizzazione calcolata sull'intero dataset (incluso il futuro).

### Implementazione degli indicatori di base

Le due funzioni di supporto calcolano gli indicatori tecnici in modo interamente causale, usando smoothing esponenziale in stile Wilder/EMA classico.


In [1]:

import numpy as np
import pandas as pd


def calculate_ema(series: pd.Series, span: int) -> pd.Series:
    """Media esponenziale causale: ogni valore dipende solo dal passato."""
    return series.astype("float64").ewm(
        span=span,
        adjust=False,      # fondamentale: adjust=False rende la EMA
                            # calcolabile in streaming, coerente con l'uso
                            # in produzione barra-per-barra
        min_periods=span,  # NaN fino a quando non c'è storico sufficiente
    ).mean()


def calculate_atr(high, low, close, window: int = 14) -> pd.Series:
    """Average True Range causale (True Range + smoothing esponenziale)."""
    previous_close = close.shift(1)  # unico accesso al passato, mai al futuro

    true_range = pd.concat(
        [
            (high - low),
            (high - previous_close).abs(),
            (low - previous_close).abs(),
        ],
        axis=1,
    ).max(axis=1)

    return true_range.ewm(
        alpha=1.0 / window,
        adjust=False,
        min_periods=window,
    ).mean()


### Costruzione dell'intero blocco di feature

La funzione principale `create_market_features` orchestra il calcolo di tutte le feature di mercato, applica la codifica ciclica del tempo, integra lo stato dell'agente e produce il flag di warm-up.

Punti di attenzione implementativi:
- **Validazione robusta in ingresso**: il DataFrame deve avere un `DatetimeIndex` ordinato cronologicamente e le colonne OHLCV richieste, altrimenti la funzione si ferma subito con un errore esplicito.
- **Controllo `High < Low`**: aggiunto dopo la revisione, per intercettare dati corrotti a monte invece di propagare `NaN`/`inf` silenziosi.
- **Contratto rigido sulle finestre (`strict_windows=True`)**: impedisce di eseguire accidentalmente la pipeline con finestre diverse da quelle della Fase 3 (`return_windows=(1,4,16)`, `ema_windows=(20,50)`), con possibilità di disattivarlo per sperimentazione futura.


In [2]:

from collections.abc import Iterable, Sequence
from dataclasses import dataclass

OHLCV_COLUMNS = ["Open", "High", "Low", "Close", "Volume"]
STATE_COLUMNS = ["current_position", "unrealized_return"]
MARKET_FEATURE_COLUMNS = [
    "return_1", "return_4", "return_16", "atr_over_close",
    "close_ema20_over_atr", "close_ema50_over_atr", "ema20_ema50_over_atr",
    "candle_body_over_range", "upper_wick_over_range", "lower_wick_over_range",
    "relative_volume", "hour_sin", "hour_cos",
]
DEFAULT_FEATURE_COLUMNS = MARKET_FEATURE_COLUMNS + STATE_COLUMNS
DEFAULT_NORMALIZATION_EXCLUDE = (
    "hour_sin", "hour_cos", "current_position", "unrealized_return",
)


def _validate_input_dataframe(dataframe: pd.DataFrame) -> None:
    if dataframe.empty:
        raise ValueError("Il dataset è vuoto.")
    if not isinstance(dataframe.index, pd.DatetimeIndex):
        raise TypeError("L'indice deve essere un DatetimeIndex.")
    if not dataframe.index.is_monotonic_increasing:
        raise ValueError("Il dataset deve essere ordinato cronologicamente.")
    missing_columns = [c for c in OHLCV_COLUMNS if c not in dataframe.columns]
    if missing_columns:
        raise ValueError(f"Colonne OHLCV mancanti: {missing_columns}")


def build_state_frame(index, state_frame=None, default_position=0.0, default_unrealized_return=0.0):
    """
    Costruisce le feature di stato dell'agente (current_position, unrealized_return).

    In assenza di un Execution Engine reale, restituisce default causali a
    zero. Quando l'Execution Engine sarà pronto, passerà un DataFrame reale
    con l'indice allineato all'OHLCV; il reindex gestisce eventuali
    disallineamenti temporali con un warning se superano l'1% delle righe.
    """
    if state_frame is None:
        return pd.DataFrame(
            {"current_position": default_position, "unrealized_return": default_unrealized_return},
            index=index, dtype="float64",
        )
    aligned = state_frame.reindex(index).copy()
    missing_fraction = aligned[["current_position", "unrealized_return"]].isna().mean().max()
    if missing_fraction > 0.01:
        print(f"[warn] reindex state_frame ha introdotto NaN su {missing_fraction:.1%} delle righe")
    for column, default_value in {
        "current_position": default_position,
        "unrealized_return": default_unrealized_return,
    }.items():
        if column not in aligned.columns:
            aligned[column] = default_value
        aligned[column] = aligned[column].fillna(default_value).astype("float64")
    return aligned[["current_position", "unrealized_return"]]


def create_market_features(
    dataframe: pd.DataFrame,
    state_frame=None,
    return_windows: Sequence[int] = (1, 4, 16),
    atr_window: int = 14,
    ema_windows: Sequence[int] = (20, 50),
    relative_volume_window: int = 20,
    drop_warmup: bool = False,
    strict_windows: bool = True,
) -> pd.DataFrame:
    """
    Genera le feature causali della Fase 3.

    Il warm-up complessivo (colonna `warmup_complete`) è dominato da EMA50
    (min_periods=50), non da relative_volume (min_periods=20) né da ATR
    (min_periods=14). Con i default correnti, warmup_complete diventa True
    dalla 50-esima barra osservata (indice 0-based: riga 49).

    Precondizione: il DataFrame in input deve provenire da
    load_market_data + validate_ohlcv (Fase 1), che garantisce prezzi OHLC
    strettamente positivi e volume non negativo.
    """
    _validate_input_dataframe(dataframe)

    if strict_windows:
        if tuple(return_windows) != (1, 4, 16):
            raise ValueError("return_windows deve essere (1, 4, 16) in modalità strict.")
        if tuple(ema_windows) != (20, 50):
            raise ValueError("ema_windows deve essere (20, 50) in modalità strict.")
    else:
        print(f"[warn] finestre non standard: return={tuple(return_windows)} ema={tuple(ema_windows)}")

    features = pd.DataFrame(index=dataframe.index)
    open_price = dataframe["Open"].astype("float64")
    high = dataframe["High"].astype("float64")
    low = dataframe["Low"].astype("float64")
    close = dataframe["Close"].astype("float64")
    volume = dataframe["Volume"].astype("float64")

    # Controllo di sicurezza: dati corrotti a monte devono bloccare la pipeline
    invalid_bars = high < low
    if invalid_bars.any():
        raise ValueError(f"Trovate {int(invalid_bars.sum())} barre con High < Low.")

    atr = calculate_atr(high=high, low=low, close=close, window=atr_window)
    ema20 = calculate_ema(close, span=ema_windows[0])
    ema50 = calculate_ema(close, span=ema_windows[1])

    bar_range = (high - low).replace(0.0, np.nan)
    safe_atr = atr.replace(0.0, np.nan)

    features["return_1"] = close.pct_change(1)
    features["return_4"] = close.pct_change(4)
    features["return_16"] = close.pct_change(16)
    features["atr_over_close"] = atr / close.replace(0.0, np.nan)
    features["close_ema20_over_atr"] = (close - ema20) / safe_atr
    features["close_ema50_over_atr"] = (close - ema50) / safe_atr
    features["ema20_ema50_over_atr"] = (ema20 - ema50) / safe_atr
    features["candle_body_over_range"] = (close - open_price) / bar_range
    features["upper_wick_over_range"] = (high - np.maximum(open_price, close)) / bar_range
    features["lower_wick_over_range"] = (np.minimum(open_price, close) - low) / bar_range
    features["relative_volume"] = volume / volume.rolling(
        window=relative_volume_window, min_periods=relative_volume_window
    ).mean()

    minute_of_day = dataframe.index.hour * 60 + dataframe.index.minute
    features["hour_sin"] = np.sin(2.0 * np.pi * minute_of_day / 1440.0)
    features["hour_cos"] = np.cos(2.0 * np.pi * minute_of_day / 1440.0)

    states = build_state_frame(index=dataframe.index, state_frame=state_frame)
    features[STATE_COLUMNS] = states[STATE_COLUMNS]

    features["warmup_complete"] = features[MARKET_FEATURE_COLUMNS].notna().all(axis=1)
    features = features[DEFAULT_FEATURE_COLUMNS + ["warmup_complete"]]

    if drop_warmup:
        features = features.loc[features["warmup_complete"]].copy()

    return features


## 3. Gestione del warm-up

Ogni indicatore rolling/esponenziale richiede un certo numero di barre prima di produrre un valore valido. Invece di riempire silenziosamente questi valori (es. con backfill, che introdurrebbe leakage), il modulo espone un flag booleano esplicito.

**Decisione progettuale:** `warmup_complete = True` solo quando **tutte** le feature di mercato sono simultaneamente disponibili. Con i parametri di default, l'indicatore più lento è EMA50 (`min_periods=50`), quindi il warm-up complessivo dura 50 barre, non 20 (relative_volume) né 14 (ATR).

Questo è stato verificato empiricamente nel test automatico:


In [3]:

# Verifica empirica del warm-up su dati sintetici
idx_demo = pd.date_range("2024-01-01", periods=120, freq="1min")
demo = pd.DataFrame({
    "Open": np.linspace(2000, 2010, 120),
    "High": np.linspace(2000.5, 2010.5, 120),
    "Low": np.linspace(1999.5, 2009.5, 120),
    "Close": np.linspace(2000.2, 2010.2, 120),
    "Volume": np.linspace(10, 20, 120),
}, index=idx_demo)

features_demo = create_market_features(demo)

print("Warm-up incompleto nelle prime 49 righe:", not features_demo["warmup_complete"].iloc[:49].any())
print("Warm-up completo dalla riga 80 in poi:", features_demo["warmup_complete"].iloc[80:].all())
print("Prima riga con warmup_complete=True:", features_demo.index[features_demo["warmup_complete"]][0])


Warm-up incompleto nelle prime 49 righe: True
Warm-up completo dalla riga 80 in poi: True
Prima riga con warmup_complete=True: 2024-01-01 00:49:00


**Uso pratico**: nella pipeline di produzione (`scripts/build_features.py`), si applica `drop_warmup=True` per rimuovere automaticamente le righe iniziali non ancora mature prima di normalizzare e filtrare le feature.


## 4. Normalizzazione — solo sul training set

### Perché è critico

Se le statistiche di normalizzazione (media, deviazione standard) venissero calcolate sull'intero dataset, la trasformazione applicata al validation/test set conterrebbe implicitamente informazione sulla loro distribuzione — una forma sottile ma reale di data leakage.

### Implementazione a due fasi

`fit_normalization_stats` calcola le statistiche **esclusivamente** sul training set. `apply_normalization` applica queste stesse statistiche (mai ricalcolate) a qualsiasi altro split.

Feature escluse dalla normalizzazione: `hour_sin`, `hour_cos` (già bounded in [-1,1] per costruzione), `current_position` (categoriale/discreta), `unrealized_return` (feature di stato, spesso con varianza degenere nei test sintetici).


In [4]:

@dataclass(frozen=True)
class NormalizationStats:
    means: pd.Series
    stds: pd.Series
    normalized_columns: tuple
    excluded_columns: tuple


def detect_missing_or_infinite(dataframe, feature_columns=None) -> pd.DataFrame:
    columns = list(feature_columns or DEFAULT_FEATURE_COLUMNS)
    subset = dataframe[columns]
    rows = []
    for column in columns:
        series = subset[column]
        missing_mask = series.isna()
        infinite_mask = np.isinf(series.to_numpy(dtype="float64", copy=False))
        if missing_mask.any() or infinite_mask.any():
            rows.append({
                "feature": column,
                "missing_count": int(missing_mask.sum()),
                "infinite_count": int(infinite_mask.sum()),
            })
    return pd.DataFrame(rows)


def assert_no_missing_or_infinite(dataframe, feature_columns=None) -> None:
    issues = detect_missing_or_infinite(dataframe, feature_columns=feature_columns)
    if not issues.empty:
        raise ValueError(f"Trovati valori mancanti o infiniti nelle feature:\n{issues.to_string(index=False)}")


def fit_normalization_stats(train_features, feature_columns=None, exclude_columns=DEFAULT_NORMALIZATION_EXCLUDE) -> NormalizationStats:
    """Calcola media e deviazione standard ESCLUSIVAMENTE sul training set."""
    columns = list(feature_columns or DEFAULT_FEATURE_COLUMNS)
    excluded = tuple(c for c in exclude_columns if c in columns)
    normalized_columns = tuple(c for c in columns if c not in excluded)
    if not normalized_columns:
        raise ValueError("Nessuna colonna disponibile per la normalizzazione.")

    reference = train_features.loc[:, normalized_columns]
    assert_no_missing_or_infinite(reference, feature_columns=normalized_columns)

    means = reference.mean(axis=0)
    stds = reference.std(axis=0, ddof=0).replace(0.0, 1.0)  # evita divisione per zero

    return NormalizationStats(means=means, stds=stds, normalized_columns=normalized_columns, excluded_columns=excluded)


def apply_normalization(features, stats: NormalizationStats) -> pd.DataFrame:
    """Applica statistiche già fittate, senza mai ricalcolarle."""
    normalized = features.copy()
    normalized.loc[:, stats.normalized_columns] = (
        normalized.loc[:, stats.normalized_columns] - stats.means
    ) / stats.stds
    return normalized


In [5]:

# Dimostrazione: normalizzazione centra correttamente il training set
features_demo2 = create_market_features(demo, drop_warmup=True)
stats_demo = fit_normalization_stats(features_demo2)
normalized_demo = apply_normalization(features_demo2, stats_demo)

check_cols = [c for c in DEFAULT_FEATURE_COLUMNS if c not in DEFAULT_NORMALIZATION_EXCLUDE]
print("Media assoluta post-normalizzazione (~0):")
print(normalized_demo[check_cols].mean().abs())
print("\nDeviazione standard post-normalizzazione (~1):")
print(normalized_demo[check_cols].std(ddof=0))


Media assoluta post-normalizzazione (~0):
return_1                  6.112481e-14
return_4                  4.247932e-14
return_16                 5.835395e-14
atr_over_close            3.144371e-13
close_ema20_over_atr      2.240774e-15
close_ema50_over_atr      1.125860e-16
ema20_ema50_over_atr      8.756689e-17
candle_body_over_range    3.939352e-04
upper_wick_over_range     6.419684e-04
lower_wick_over_range     0.000000e+00
relative_volume           3.455452e-14
dtype: float64

Deviazione standard post-normalizzazione (~1):
return_1                  1.0
return_4                  1.0
return_16                 1.0
atr_over_close            1.0
close_ema20_over_atr      1.0
close_ema50_over_atr      1.0
ema20_ema50_over_atr      1.0
candle_body_over_range    1.0
upper_wick_over_range     1.0
lower_wick_over_range     0.0
relative_volume           1.0
dtype: float64


## 5. Verifica di valori infiniti o mancanti

Due funzioni dedicate separano la fase diagnostica da quella di enforcement:
- `detect_missing_or_infinite`: restituisce una tabella con conteggio missing/infiniti per ciascuna feature, utile per debug e logging.
- `assert_no_missing_or_infinite`: solleva un errore bloccante se trova anomalie, da usare come guardia nella pipeline di produzione e nei test.

**Nota importante**: questi controlli vanno eseguiti **dopo** aver filtrato `warmup_complete` (o con `drop_warmup=True`), perché i `NaN` nelle prime barre sono un comportamento atteso del warm-up causale, non un bug.

Nella pipeline reale (`build_features.py`), questo controllo è stato eseguito su ~6.48M barre reali (2004-2026) senza alcuna anomalia riscontrata, confermando la robustezza numerica del modulo su tutto lo storico.


## 6. Eliminazione delle feature fortemente correlate

### Logica implementata

`drop_highly_correlated_features` calcola la matrice di correlazione di Pearson (in valore assoluto) **esclusivamente sul training set**, poi scarta deterministicamente la feature "più a destra" nell'ordine fornito quando una coppia supera la soglia (default 0.95).

Feature protette da default (mai scartate anche se correlate): `hour_sin`, `hour_cos`, `current_position`, `unrealized_return`, per la loro semantica strutturale.

**Limite noto e documentato**: il metodo rileva solo dipendenze lineari; relazioni non lineari (es. tra volatilità e volume relativo in regimi di stress) non vengono captate. Estensione futura possibile con correlazione di Spearman o mutual information.


In [6]:

@dataclass(frozen=True)
class CorrelationFilterResult:
    kept_columns: tuple
    dropped_columns: tuple
    drop_reasons: tuple


def drop_highly_correlated_features(
    train_features, feature_columns=None, threshold: float = 0.95,
    protected_columns: Iterable[str] = ("hour_sin", "hour_cos", "current_position", "unrealized_return"),
) -> CorrelationFilterResult:
    """
    Rimuove feature con correlazione di Pearson (lineare) >= threshold,
    calcolata esclusivamente sul training set.
    """
    ordered_columns = list(feature_columns or DEFAULT_FEATURE_COLUMNS)
    protected = set(protected_columns)
    candidate_columns = [c for c in ordered_columns if c not in protected]

    if len(candidate_columns) < 2:
        return CorrelationFilterResult(kept_columns=tuple(ordered_columns), dropped_columns=tuple(), drop_reasons=tuple())

    correlation_matrix = train_features.loc[:, candidate_columns].corr().abs()
    dropped = set()
    reasons = []
    for i, left in enumerate(candidate_columns):
        if left in dropped:
            continue
        for right in candidate_columns[i + 1:]:
            if right in dropped:
                continue
            value = correlation_matrix.loc[left, right]
            if pd.isna(value):
                continue
            if value >= threshold:
                dropped.add(right)
                reasons.append((left, right, float(value)))

    kept_columns = tuple(c for c in ordered_columns if c not in dropped)
    return CorrelationFilterResult(
        kept_columns=kept_columns,
        dropped_columns=tuple(c for c in ordered_columns if c in dropped),
        drop_reasons=tuple(reasons),
    )


### Risultato sul dataset reale (XAU/USD M1, 2004-2026)

Sull'intero training set reale (~4.45M barre), la correlazione massima misurata tra le 11 feature di mercato normalizzate è risultata **0.898** (tra `close_ema20_over_atr` e `close_ema50_over_atr`), ben sotto la soglia 0.95. Di conseguenza, **nessuna feature è stata scartata** sul dataset reale.

Le coppie più vicine alla soglia, in ordine decrescente:

| Coppia di feature | Correlazione assoluta |
|---|---|
| `close_ema20_over_atr` vs `close_ema50_over_atr` | 0.898 |
| `close_ema50_over_atr` vs `ema20_ema50_over_atr` | 0.838 |
| `return_16` vs `close_ema20_over_atr` | 0.710 |
| `return_16` vs `close_ema50_over_atr` | 0.672 |
| `return_1` vs `candle_body_over_range` | 0.609 |

Questo risultato è stato validato calcolando manualmente la matrice di correlazione sul parquet di training salvato, confermando che il filtro non presenta bug e che il dataset reale è naturalmente meno ridondante rispetto al dataset sintetico usato nei test automatici (dove 3 feature venivano scartate per costruzione deliberata del test).

**Nota per iterazioni future**: se si desidera un set di feature più parsimonioso in vista del training RL, si può valutare di abbassare la soglia a 0.85, che scarterebbe la coppia EMA20/EMA50 su ATR.


In [7]:

# Riproduzione del controllo eseguito sul parquet di training reale
# (adattare il path al proprio ambiente locale)
#
# df = pd.read_parquet("data/processed/xauusd_m1_train.parquet")
# cols = ["return_1","return_4","return_16","atr_over_close","close_ema20_over_atr",
#         "close_ema50_over_atr","ema20_ema50_over_atr","candle_body_over_range",
#         "upper_wick_over_range","lower_wick_over_range","relative_volume"]
# corr = df[cols].corr().abs()
# corr_vals = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
# pairs = corr_vals.stack().sort_values(ascending=False)
# print(pairs.head(10))

# Risultato ottenuto in produzione (per riferimento):
risultato_reale = {
    ("close_ema20_over_atr", "close_ema50_over_atr"): 0.897778,
    ("close_ema50_over_atr", "ema20_ema50_over_atr"): 0.838448,
    ("return_16", "close_ema20_over_atr"): 0.710183,
    ("return_16", "close_ema50_over_atr"): 0.671685,
    ("return_1", "candle_body_over_range"): 0.609423,
    ("return_4", "close_ema20_over_atr"): 0.609112,
    ("close_ema20_over_atr", "ema20_ema50_over_atr"): 0.512703,
    ("return_1", "return_4"): 0.501021,
    ("return_4", "return_16"): 0.497889,
    ("return_4", "close_ema50_over_atr"): 0.438815,
}
for pair, value in risultato_reale.items():
    print(f"{pair}: {value:.4f}")
print("\nMassimo:", max(risultato_reale.values()), "< soglia 0.95 -> nessuna feature scartata")


('close_ema20_over_atr', 'close_ema50_over_atr'): 0.8978
('close_ema50_over_atr', 'ema20_ema50_over_atr'): 0.8384
('return_16', 'close_ema20_over_atr'): 0.7102
('return_16', 'close_ema50_over_atr'): 0.6717
('return_1', 'candle_body_over_range'): 0.6094
('return_4', 'close_ema20_over_atr'): 0.6091
('close_ema20_over_atr', 'ema20_ema50_over_atr'): 0.5127
('return_1', 'return_4'): 0.5010
('return_4', 'return_16'): 0.4979
('return_4', 'close_ema50_over_atr'): 0.4388

Massimo: 0.897778 < soglia 0.95 -> nessuna feature scartata


## 7. Test future-append: verifica del criterio di completamento

### Il criterio

> Aggiungere dati futuri non deve modificare le feature già costruite per il passato.

Questo è il controllo più importante della Fase 3, perché una violazione qui significherebbe leakage strutturale non rilevabile con semplici controlli di missing/infiniti.

### Implementazione

Il modulo `leakage_checks.py` fornisce due asserzioni:

1. **`assert_future_append_invariant`**: costruisce le feature sullo storico, poi le ricostruisce su storico+futuro concatenato, e verifica che le righe del passato coincidano entro tolleranze numeriche (`rtol=1e-9`, `atol=1e-12`).

2. **`assert_past_normalization_invariant`**: verifica la stessa invarianza anche dopo l'applicazione della normalizzazione, per assicurarsi che il riutilizzo delle statistiche fittate sul train non introduca discrepanze quando si estende il dataset.


In [8]:

from collections.abc import Callable
from pandas.testing import assert_frame_equal

FeatureBuilder = Callable[[pd.DataFrame], pd.DataFrame]


def assert_future_append_invariant(
    feature_builder: FeatureBuilder,
    history_dataframe: pd.DataFrame,
    future_dataframe: pd.DataFrame,
    feature_columns=None,
    rtol: float = 1e-9,
    atol: float = 1e-12,
) -> None:
    """Verifica che le feature del passato non cambino aggiungendo dati futuri."""
    past_features = feature_builder(history_dataframe)

    appended_dataframe = pd.concat(
        [history_dataframe, future_dataframe], axis=0
    ).sort_index(kind="stable")

    appended_features = feature_builder(appended_dataframe)
    compare_columns = list(feature_columns or past_features.columns)

    assert_frame_equal(
        past_features.loc[:, compare_columns],
        appended_features.loc[past_features.index, compare_columns],
        check_dtype=False, check_exact=False, rtol=rtol, atol=atol,
    )


def assert_past_normalization_invariant(
    past_features, appended_features, normalized_past, normalized_appended,
    feature_columns, rtol: float = 1e-9, atol: float = 1e-12,
) -> None:
    """Verifica l'invarianza sia sulle feature raw sia su quelle normalizzate."""
    assert_frame_equal(
        past_features.loc[:, feature_columns],
        appended_features.loc[past_features.index, feature_columns],
        check_dtype=False, check_exact=False, rtol=rtol, atol=atol,
    )
    assert_frame_equal(
        normalized_past.loc[:, feature_columns],
        normalized_appended.loc[past_features.index, feature_columns],
        check_dtype=False, check_exact=False, rtol=rtol, atol=atol,
    )


In [9]:

# Dimostrazione pratica del test future-append
idx_full = pd.date_range("2024-03-01", periods=260, freq="1min")
base = 2050.0 + np.linspace(0.0, 18.0, 260)
seasonal = 0.9 * np.sin(np.arange(260) / 8.0)
close_full = base + seasonal
open_full = close_full - 0.12 * np.cos(np.arange(260) / 6.0)
high_full = np.maximum(open_full, close_full) + 0.2
low_full = np.minimum(open_full, close_full) - 0.2
volume_full = 120.0 + (np.arange(260) % 17) + 0.25 * np.cos(np.arange(260) / 10.0)

full_dataframe = pd.DataFrame({
    "Open": open_full, "High": high_full, "Low": low_full,
    "Close": close_full, "Volume": volume_full,
}, index=idx_full)

history = full_dataframe.iloc[:180].copy()
future = full_dataframe.iloc[180:].copy()

assert_future_append_invariant(
    feature_builder=lambda frame: create_market_features(frame),
    history_dataframe=history,
    future_dataframe=future,
)
print("OK: le feature del passato non cambiano aggiungendo dati futuri.")


OK: le feature del passato non cambiano aggiungendo dati futuri.


## 8. Test automatici implementati

### `tests/test_features.py` — 6 test

| Test | Cosa verifica |
|---|---|
| `test_selected_feature_columns_exist` | Le colonne prodotte corrispondono esattamente alla selezione richiesta |
| `test_warmup_flag_becomes_true_after_indicator_history` | Il warm-up è incompleto nelle prime 49 righe e completo dopo la riga 80 |
| `test_missing_and_infinite_checks_ignore_warmup_after_drop` | Nessun missing/infinito dopo `drop_warmup=True` |
| `test_training_only_normalization_centers_training_split` | Media ≈ 0 e deviazione standard ≈ 1 dopo normalizzazione sul train |
| `test_correlation_filter_drops_later_duplicate_feature` | Una feature clonata (correlazione ≈ 1) viene correttamente scartata |
| `test_strict_feature_contract_rejects_wrong_phase_windows` | Finestre diverse da quelle di fase generano `ValueError` in modalità strict |

### `tests/test_no_leakage.py` — 2 test

| Test | Cosa verifica |
|---|---|
| `test_future_append_does_not_change_past_features` | Criterio di completamento sulle feature raw |
| `test_future_append_does_not_change_past_normalized_values_when_train_stats_are_reused` | Criterio di completamento anche dopo normalizzazione con statistiche riusate |

**Risultato dell'esecuzione** (`pytest tests/test_features.py tests/test_no_leakage.py -v`):

```
8 passed in 3.25s
```

Tutti gli 8 test sono stati eseguiti con successo nell'ambiente Windows/Python 3.12 del progetto.


## 9. Pipeline end-to-end: `scripts/build_features.py`

Lo script collega tutti i moduli precedenti in un'unica pipeline eseguibile, riutilizzando `load_market_data` (Fase 1) e `chronological_date_split`.

### Passaggi della pipeline

1. **Caricamento** del CSV grezzo M1 tramite `load_market_data`, con parametri letti da `config.yaml`.
2. **Calcolo dei punti di split** cronologico in base a `train_ratio` e `validation_ratio` configurati (0.70 / 0.15 / 0.15).
3. **Split cronologico** in train/validation/test senza sovrapposizioni, tramite `chronological_date_split` (corretto il bug preesistente `return train, validation, tests` → `test`).
4. **Costruzione delle feature** separatamente su ciascuno split, con `drop_warmup=True`.
5. **Verifica missing/infiniti** su ciascuno split con `assert_no_missing_or_infinite`.
6. **Fit della normalizzazione** esclusivamente sul training set.
7. **Applicazione della normalizzazione** a tutti e tre gli split con le stesse statistiche.
8. **Filtro di correlazione** calcolato sul train normalizzato, applicato identicamente a validation e test.
9. **Salvataggio** di tre file parquet (`xauusd_m1_train.parquet`, `_validation.parquet`, `_test.parquet`) e due report JSON (`normalization_stats.json`, `feature_correlation.json`).

### Motivo della separazione per split al passo 4

Le feature vengono calcolate separatamente su train, validation e test (e non sull'intero dataset prima di splittare) per evitare che il warm-up di un indicatore a inizio validation "veda" dati di fine training attraverso rolling/ewm, il che violerebbe la separazione cronologica tra split.


## 10. Esecuzione sul dataset reale

### Comando eseguito

```powershell
python .\scripts\build_features.py
```

### Output ottenuto

```
Caricamento dataset sorgente: ...data\raw\XAU_1m_data.csv
Split cronologico -> train_end=2020-03-12 01:36:00 validation_end=2023-02-02 12:59:00
Costruzione feature causali per ciascuno split...
Verifica assenza di missing/infiniti dopo il warm-up...
Fit della normalizzazione SOLO sul training set...
Filtro delle feature fortemente correlate (calcolato sul train)...
Feature scartate per correlazione: ()

Completato.
Train: (4445948, 15), Validation: (1016859, 15), Test: (1018239, 15)
```

### Verifica delle proporzioni di split

Sul totale di 6.481.046 barre M1 valide (dopo il warm-up), le proporzioni effettive risultano molto vicine a quelle configurate (70% / 15% / 15%):


In [10]:

totale = 4445948 + 1016859 + 1018239
print("Righe totali dopo warm-up:", totale)
print(f"Train: {4445948/totale:.4%}")
print(f"Validation: {1016859/totale:.4%}")
print(f"Test: {1018239/totale:.4%}")


Righe totali dopo warm-up: 6481046
Train: 68.5992%
Validation: 15.6897%
Test: 15.7110%


Le percentuali osservate (68.6% / 15.7% / 15.7%) sono coerenti con quelle configurate (70% / 15% / 15%): la piccola differenza è dovuta alle righe di warm-up scartate all'inizio di ciascuno split, che riducono leggermente la dimensione del training set rispetto agli altri due (essendo un unico blocco di warm-up per split, l'effetto relativo è più visibile sul train solo in termini assoluti, non percentuali significativi).

### Nessuna anomalia numerica

Nessun errore è stato generato dai controlli `assert_no_missing_or_infinite` sui quasi 6.5 milioni di barre reali (2004-2026), confermando la robustezza del calcolo causale delle feature su tutto lo storico disponibile, inclusi weekend, chiusure prolungate e periodi di alta volatilità.


## 11. Sintesi finale della Fase 3

| Attività richiesta | Stato | Evidenza |
|---|---|---|
| Calcolare le feature causalmente | ✅ Completato | Solo `shift`, `pct_change`, `rolling(min_periods=...)`, `ewm(adjust=False, min_periods=...)` |
| Gestire il warm-up degli indicatori | ✅ Completato | Flag `warmup_complete`, verificato a riga 49 su dati sintetici |
| Normalizzare usando solo il training set | ✅ Completato | `fit_normalization_stats` + `apply_normalization`, media≈0 e std≈1 verificati |
| Verificare valori infiniti o mancanti | ✅ Completato | `assert_no_missing_or_infinite`, nessuna anomalia su 6.48M barre reali |
| Eliminare feature fortemente correlate | ✅ Completato | Soglia 0.95, massimo osservato 0.898 sul train reale, nessuna feature scartata |
| Implementare il test future-append | ✅ Completato | `assert_future_append_invariant`, 8/8 test passati |

### Criterio di completamento verificato

> Aggiungere dati futuri non deve modificare le feature già costruite per il passato.

Verificato sia strutturalmente (nessuna operazione guarda-avanti nel codice) sia empiricamente (test automatici passati e pipeline eseguita con successo sul dataset reale completo).

### Prossimi passi

1. Valutare se abbassare la soglia di correlazione (es. 0.85) per un set di feature più parsimonioso.
2. Progettare l'Execution Engine, che dovrà popolare `current_position` e `unrealized_return` con lo stato reale dell'agente al posto degli stub a zero attuali.
3. Chiarire se il timeframe di lavoro per le fasi successive resta M1 o passa a M15 (già disponibile come parquet resample-ato nel repository), poiché questo influenza il significato di `hour_sin`/`hour_cos` e la durata reale del warm-up in tempo di mercato.
